In [40]:
!pip install pyotp mplfinance pandas_ta fyers_apiv3 pygame stable-baselines3 gymnasium shimmy optuna

In [41]:
import requests
import base64
from datetime import datetime, timedelta, date
from datetime import time as dt_time
import time
import threading
import pyotp
from pytz import timezone
import pandas as pd
import numpy as np
from urllib.parse import urlparse, parse_qs
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter
import mplfinance as mpf
import pandas_ta as ta
import pygame
import os
import pytz
import json
import re
import random
import pickle

import gymnasium as gym
from gymnasium import spaces
from IPython.display import display, clear_output
from tqdm.notebook import tqdm

from fyers_apiv3 import fyersModel
from fyers_apiv3.FyersWebsocket import data_ws

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Dense, LSTM, GRU, Dropout, Attention, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.callbacks import Callback, ModelCheckpoint
from scipy.signal import argrelextrema
import tensorflow as tf
from collections import deque
from tensorflow import keras
from tensorflow.keras import layers, optimizers, models
from scipy.signal import find_peaks

In [42]:
def get_index_symbol_and_quantity(index):

    # Dictionary mapping index name to index symbols
    index_symbols = {
        'Bankex': 'BSE:BANKEX-INDEX',
        'Finnifty': 'NSE:FINNIFTY-INDEX',
        'Bank Nifty': 'NSE:NIFTYBANK-INDEX',
        'Nifty': 'NSE:NIFTY50-INDEX',
        'Sensex': 'BSE:SENSEX-INDEX'
    }

    # Determine the index symbol for the given index
    index_symbol = index_symbols.get(index, 'Invalid Index')

    # Determine the quantity based on the index symbol
    if index_symbol == "NSE:NIFTY50-INDEX":
        quantity = 25  # 25 is one lot for Nifty
    elif index_symbol == "NSE:NIFTYBANK-INDEX":
        quantity = 15  # 15 is one lot for Bank Nifty
    elif index_symbol == "NSE:FINNIFTY-INDEX":
        quantity = 40  # 40 is one lot for Finnifty
    elif index_symbol == "BSE:SENSEX-INDEX":
        quantity = 20  # 20 is two lot for Sensex
    elif index_symbol == "BSE:BANKEX-INDEX":
        quantity = 15  # 15 is one lot for Bankex
    else:
        quantity = 0  # Default value if none of the conditions match

    return index_symbol, quantity

In [43]:
app_id = "TS79V3NXK1-100"
secret_key = "KQCPB0FJ74"
redirect_uri = "https://google.com"
fyers_user = "XM22383"
fyers_pin = "4628"
fyers_totp = "EAQD6K4IUYOEGPJNVE6BMPTUSDCWIOHW"
response_type = "code"
state = "sample_state"
grant_type = "authorization_code"

fyers = None
fyers_socket = None

index_symbol, quantity = get_index_symbol_and_quantity("Bank Nifty")

interval_minutes = 2 # Set the interval to 1, 5, or 15 minutes

ist_timezone = pytz.timezone("Asia/Kolkata")

# Initialize the mixer
#pygame.mixer.init()

# Load the sound file
#pygame.mixer.music.load('sounds/alert.mp3')

#Variables
ce_ltp = 0
pe_ltp = 0
index_ltp = 0
buy_sell_checked = False
ce_strike = None
pe_strike = None
ce_symbol = None
pe_symbol = None

target = 80
trailing_sl = 40

brokerage = 100

fixed_ltp = 0
fixed_index_ltp = 0
prev_ltp = 0
target_inside = 0
target_index_inside = 0
trailing_sl_inside = 0
trailing_index_inside = 0

active_order = False

sl_hit_condition = False
total_loss = 0
total_profit = 0
overall_win = 0
overall_loss = 0
total_points = 0

unsubscribe_done = False

active_order_sleep = 1

In [44]:
 session = fyersModel.SessionModel(
    client_id=app_id,
    secret_key=secret_key,
    redirect_uri=redirect_uri,
    response_type=response_type,
    grant_type=grant_type
)

def getEncodedString(string):
    string = str(string)
    base64_bytes = base64.b64encode(string.encode("ascii"))
    return base64_bytes.decode("ascii")

if session is not None:
    session.generate_authcode()

    url_send_login_otp = "https://api-t2.fyers.in/vagator/v2/send_login_otp_v2"
    res = requests.post(url=url_send_login_otp, json={"fy_id": getEncodedString(fyers_user), "app_id": "2"}).json()

    if datetime.now().second % 30 > 27:
        time.sleep(5)

    url_verify_otp = "https://api-t2.fyers.in/vagator/v2/verify_otp"
    res2 = requests.post(url=url_verify_otp, json={"request_key": res["request_key"], "otp": pyotp.TOTP(fyers_totp).now()}).json()

    ses = requests.Session()
    url_verify_otp2 = "https://api-t2.fyers.in/vagator/v2/verify_pin_v2"
    payload2 = {"request_key": res2["request_key"], "identity_type": "pin", "identifier": getEncodedString(fyers_pin)}
    res3 = ses.post(url=url_verify_otp2, json=payload2).json()

    ses.headers.update({'authorization': f"Bearer {res3['data']['access_token']}"})

    tokenurl = "https://api-t1.fyers.in/api/v3/token"
    payload3 = {
        "fyers_id": fyers_user,
        "app_id": app_id[:-4],
        "redirect_uri": redirect_uri,
        "appType": "100",
        "code_challenge": "",
        "state": "None",
        "scope": "",
        "nonce": "",
        "response_type": "code",
        "create_cookie": True
    }

    res3 = ses.post(url=tokenurl, json=payload3).json()

    url = res3['Url']
    parsed = urlparse(url)
    auth_code = parse_qs(parsed.query)['auth_code'][0]

    session.set_token(auth_code)

    auth_response = session.generate_token()
    access_token = auth_response["access_token"]

    fyers = fyersModel.FyersModel(client_id=app_id, token=access_token)

    ws_token = app_id + ":" + access_token
    fyers_socket = data_ws.FyersDataSocket(access_token=ws_token, log_path="")

pd.DataFrame(fyers.get_profile())

,s,code,message,data
fy_id,ok,200,,XM22383
name,ok,200,,MARSHAL TUDU
image,ok,200,,https://myaccount-docs-prod.fyers.in/Profile_P...
display_name,ok,200,,None
pin_change_date,ok,200,,25-09-2023 17:16:16
email_id,ok,200,,iammarshal22@gmail.com
pwd_change_date,ok,200,,01-06-2022 20:36:31
PAN,ok,200,,---------
mobile_number,ok,200,,8458060663
totp,ok,200,,True


In [45]:
def fetch_candle_data(number):
    while True:
        try:
            today = date.today()
            yesterday = today - timedelta(number)

            data = {
                "symbol": index_symbol,
                "resolution": interval_minutes,
                "date_format": "1",
                "range_from": yesterday,
                "range_to": today,
                "cont_flag": "1"
            }

            result = fyers.history(data=data)

            if result is not None:
                train_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                return train_df
        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [46]:
def fetch_train_candle_data(days_count):
    train_df = pd.DataFrame()

    while True:
        try:
            date_increment = 100
            for i in range(days_count):
                today = date.today() - timedelta(date_increment)
                yesterday = today - timedelta(100)

                data = {
                    "symbol": index_symbol,
                    "resolution": interval_minutes,
                    "date_format": "1",
                    "range_from": yesterday,
                    "range_to": today,
                    "cont_flag": "1"
                }

                result = fyers.history(data=data)

                if result is not None:
                    temp_df = pd.DataFrame(result['candles'], columns=['datetime', 'open', 'high', 'low', 'close', 'volume'])
                    train_df = pd.concat([temp_df, train_df], ignore_index=True)

                date_increment += 100

            if train_df is not None:
                return train_df

        except Exception as e:
            print(f"Error fetching Candle Data: {e}")
            time.sleep(active_order_sleep)

In [47]:
train_df = fetch_candle_data(100)

#train_df = fetch_train_candle_data(10)

print(len(train_df))

train_df = train_df.drop_duplicates(subset='datetime', keep='first')

print(len(train_df))

12815
12815


In [48]:

class FullFeaturePipeline:
    def __init__(self, df: pd.DataFrame, base_interval_minutes: int):
        """
        df: DataFrame with columns [datetime, open, high, low, close, volume (optional)]
            'datetime' is a Unix timestamp in seconds.
        base_interval_minutes: The base timeframe in minutes (e.g. 2, 5, 15, etc.).
        """
        self.df = df.copy()
        self.base_interval = base_interval_minutes

        self.htf_map = {
            "15min":  15,
            "30min":  30,
            "1h":     60,
            "1d":     1440,
        }

    def preprocess_datetime(self):
        """
        1) Convert Unix timestamp to local time (Asia/Kolkata).
        2) Check for duplicates/missing timestamps.
        3) Sort by time and set 'datetime' as the index.
        """
        ist = timezone('Asia/Kolkata')

        self.df['datetime'] = pd.to_datetime(self.df['datetime'], unit='s')
        self.df['datetime'] = (self.df['datetime']
                               .dt.tz_localize('UTC')
                               .dt.tz_convert(ist)
                               .dt.tz_localize(None))

        if self.df['datetime'].duplicated().any() or self.df['datetime'].isnull().any():
            raise ValueError("The 'datetime' column contains duplicates or missing values.")

        self.df.sort_values('datetime', inplace=True)
        self.df.set_index('datetime', inplace=True)
        return self

    def clean_data(self):
        """
        Optionally drop 'volume' column if it contains zeros or NaNs.
        Adjust if you prefer different volume handling.
        """
        if 'volume' in self.df.columns:
            if self.df['volume'].isnull().any() or (self.df['volume'] == 0).any():
                self.df.drop('volume', axis=1, inplace=True, errors='ignore')
        return self

    def add_base_timeframe_features(self):
        """
        Compute technical indicators on the base timeframe:
        MACD, RSI, Bollinger Bands, ATR, etc., using pandas_ta.
        """
        self.df.sort_index(inplace=True)

        # RSI on base timeframe close
        self.df['rsi_base'] = ta.rsi(self.df['close'], length=14)

        # MACD
        macd = ta.macd(self.df['close'], fast=12, slow=26, signal=9)
        self.df['macd'] = macd['MACD_12_26_9']
        self.df['macd_h'] = macd['MACDh_12_26_9']
        self.df['macd_s'] = macd['MACDs_12_26_9']

        # Bollinger Bands
        bbands = ta.bbands(self.df['close'], length=20, std=2)
        self.df['bb_lower'] = bbands['BBL_20_2.0']
        self.df['bb_mid'] = bbands['BBM_20_2.0']
        self.df['bb_upper'] = bbands['BBU_20_2.0']

        # ATR
        self.df['atr_base'] = ta.atr(self.df['high'], self.df['low'], self.df['close'], length=14)

        # Example target & stop based on ATR multiples
        self.df['Target'] = 2 * self.df['atr_base']
        self.df['StopLoss'] = 1 * self.df['atr_base']

        return self

    def create_higher_timeframes(self):
        """
        Resample up to 1D, compute aggregates + an example RSI,
        then forward-fill onto base timeframe index.
        """
        self.df.sort_index(inplace=True)

        valid_htfs = [(rule, mins) for rule, mins in self.htf_map.items()
                      if mins > self.base_interval]

        agg_dict = {
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
        }
        if 'volume' in self.df.columns:
            agg_dict['volume'] = 'sum'

        for rule, total_mins in valid_htfs:
            df_htf = self.df.resample(rule).agg(agg_dict).dropna()

            # Compute RSI for higher timeframe
            df_htf[f'rsi_{rule}'] = ta.rsi(df_htf['close'], length=14)

            # Forward-fill
            df_htf_ff = df_htf.reindex(self.df.index, method='ffill')

            # Rename columns and join
            for col in df_htf_ff.columns:
                if col in agg_dict:
                    new_col = f"{col}_{rule}"
                else:
                    new_col = col
                self.df[new_col] = df_htf_ff[col]

        return self

    def label_signals(self):
        """
        Generate buy/sell signals based on target and stop-loss levels.
        This replaces the old 'generate_labels' approach.
        """
        # Initialize columns
        self.df['Signal'] = 0
        self.df['Entry Price'] = 0.0
        self.df['Exit Price'] = 0.0

        for i in range(len(self.df)):
            entry_price = self.df['close'].iloc[i]

            target = self.df['Target'].iloc[i]
            stop_loss = self.df['StopLoss'].iloc[i]

            buy_target_price = entry_price + target
            buy_sl_price = entry_price - stop_loss

            sell_target_price = entry_price - target
            sell_sl_price = entry_price + stop_loss

            # Look ahead from the next candle onward
            future_data = self.df.iloc[i + 1:]

            # --- Buy scenario ---
            for j in range(len(future_data)):
                future_high = future_data['high'].iloc[j]
                future_low = future_data['low'].iloc[j]

                # If the future high crosses above our buy target -> Buy signal
                if future_high >= buy_target_price:
                    self.df.at[self.df.index[i], 'Signal'] = 1
                    self.df.at[self.df.index[i], 'Entry Price'] = entry_price
                    self.df.at[self.df.index[i], 'Exit Price'] = future_high
                    break
                # If the future low crosses below our buy stop -> no buy
                elif future_low <= buy_sl_price:
                    break

            # --- Sell scenario ---
            for j in range(len(future_data)):
                future_high = future_data['high'].iloc[j]
                future_low = future_data['low'].iloc[j]

                # If the future low crosses below our sell target -> Sell signal
                if future_low <= sell_target_price:
                    self.df.at[self.df.index[i], 'Signal'] = 2
                    self.df.at[self.df.index[i], 'Entry Price'] = entry_price
                    self.df.at[self.df.index[i], 'Exit Price'] = future_low
                    break
                # If the future high crosses above our sell stop -> no sell
                elif future_high >= sell_sl_price:
                    break

        return self

    def run_pipeline(self):
        """
        Run all standard pipeline steps.
        You can choose whether or not to call label_signals here,
        depending on if you want signals as part of the default run.
        """
        (self.preprocess_datetime()
             .clean_data()
             .add_base_timeframe_features()
             .create_higher_timeframes())

        return self

    def get_processed_df(self):
        """
        Retrieve the final DataFrame (including any signals if label_signals() was called).
        """
        # Drop rows that have become NaN or the initial lookbacks
        self.df.dropna(axis=0, how='any', inplace=True)
        self.df = self.df[[col for col in self.df.columns if col not in ['Entry Price', 'Exit Price']]]

        return self.df

train_pipeline = FullFeaturePipeline(train_df, interval_minutes)
train_pipeline.run_pipeline()
train_pipeline.get_processed_df()

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,open_1h,high_1h,low_1h,close_1h,rsi_1h,open_1d,high_1d,low_1d,close_1d,rsi_1d
datetime,,,,,,,,,,,,,,,,,,,,,
2024-11-01 17:59:00,51550.15,51704.85,51459.40,51704.75,83.024394,55.179578,18.113710,37.065868,51322.329024,51480.5025,...,51550.15,51704.85,51459.4,51704.75,51.968670,51550.15,51825.50,51459.40,51675.2,49.544703
2024-11-01 18:01:00,51702.25,51755.45,51673.25,51750.30,85.054114,67.805492,24.591699,43.213793,51305.207638,51497.9125,...,51702.25,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703
2024-11-01 18:03:00,51745.95,51772.05,51728.40,51746.40,84.126633,76.613766,26.719978,49.893787,51302.608411,51516.0500,...,51702.25,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703
2024-11-01 18:05:00,51746.90,51759.55,51718.45,51719.60,77.844693,80.503852,24.488052,56.015800,51309.282945,51532.3650,...,51702.25,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703
2024-11-01 18:07:00,51723.45,51765.55,51714.05,51762.05,80.347900,86.020542,24.003794,62.016749,51314.193536,51550.2725,...,51702.25,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-21 15:21:00,48600.10,48619.60,48590.35,48597.50,51.151881,-9.366393,16.112085,-25.478478,48453.524064,48543.4125,...,48519.40,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964
2025-01-21 15:23:00,48595.55,48611.30,48592.95,48600.35,51.600108,-6.991569,14.789527,-21.781096,48463.616940,48549.9675,...,48519.40,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964
2025-01-21 15:25:00,48600.95,48618.50,48594.90,48606.50,52.610632,-4.560678,13.776334,-18.337013,48471.167848,48555.9050,...,48519.40,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964


In [49]:
train_pipeline.get_processed_df().columns

Index(['open', 'high', 'low', 'close', 'rsi_base', 'macd', 'macd_h', 'macd_s',
       'bb_lower', 'bb_mid', 'bb_upper', 'atr_base', 'Target', 'StopLoss',
       'open_15min', 'high_15min', 'low_15min', 'close_15min', 'rsi_15min',
       'open_30min', 'high_30min', 'low_30min', 'close_30min', 'rsi_30min',
       'open_1h', 'high_1h', 'low_1h', 'close_1h', 'rsi_1h', 'open_1d',
       'high_1d', 'low_1d', 'close_1d', 'rsi_1d'],
      dtype='object')

In [50]:
train_pipeline.label_signals()

train_pipeline.get_processed_df()

,open,high,low,close,rsi_base,macd,macd_h,macd_s,bb_lower,bb_mid,...,high_1h,low_1h,close_1h,rsi_1h,open_1d,high_1d,low_1d,close_1d,rsi_1d,Signal
datetime,,,,,,,,,,,,,,,,,,,,,
2024-11-01 17:59:00,51550.15,51704.85,51459.40,51704.75,83.024394,55.179578,18.113710,37.065868,51322.329024,51480.5025,...,51704.85,51459.4,51704.75,51.968670,51550.15,51825.50,51459.40,51675.2,49.544703,1
2024-11-01 18:01:00,51702.25,51755.45,51673.25,51750.30,85.054114,67.805492,24.591699,43.213793,51305.207638,51497.9125,...,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703,0
2024-11-01 18:03:00,51745.95,51772.05,51728.40,51746.40,84.126633,76.613766,26.719978,49.893787,51302.608411,51516.0500,...,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703,0
2024-11-01 18:05:00,51746.90,51759.55,51718.45,51719.60,77.844693,80.503852,24.488052,56.015800,51309.282945,51532.3650,...,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703,1
2024-11-01 18:07:00,51723.45,51765.55,51714.05,51762.05,80.347900,86.020542,24.003794,62.016749,51314.193536,51550.2725,...,51825.50,51631.3,51675.20,51.051880,51550.15,51825.50,51459.40,51675.2,49.544703,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-01-21 15:21:00,48600.10,48619.60,48590.35,48597.50,51.151881,-9.366393,16.112085,-25.478478,48453.524064,48543.4125,...,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964,0
2025-01-21 15:23:00,48595.55,48611.30,48592.95,48600.35,51.600108,-6.991569,14.789527,-21.781096,48463.616940,48549.9675,...,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964,0
2025-01-21 15:25:00,48600.95,48618.50,48594.90,48606.50,52.610632,-4.560678,13.776334,-18.337013,48471.167848,48555.9050,...,48639.30,48505.9,48626.10,43.690624,49532.00,49543.15,48430.95,48626.1,38.104964,0


In [51]:
# Count the occurrences of each signal in the DataFrame
signal_counts = train_pipeline.get_processed_df()['Signal'].value_counts()

signal_counts

,count
Signal,
2,3519
1,3377
0,3287


In [52]:
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import MinMaxScaler

############################################
# 1. DATASET & PROCESSING
############################################

class TimeSeriesDataset(Dataset):
    """
    PyTorch Dataset for (X, y) pairs, where:
      - X: shape [N, seq_len, num_features]
      - y: shape [N] with integer labels in {0,1,2}
    """
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def create_sliding_windows(df, seq_len=128, feature_cols=None, label_col='label'):
    """
    Creates sliding windows of length seq_len from the DataFrame `df`.
    Returns:
        X: shape [num_samples, seq_len, num_features]
        y: shape [num_samples,]
    """
    if feature_cols is None:
        # By default, take all columns except the label
        feature_cols = [c for c in df.columns if c != label_col]

    data = df[feature_cols].values  # shape [N, num_features]
    labels = df[label_col].values   # shape [N, ]

    X, y = [], []
    for i in range(len(df) - seq_len):
        X.append(data[i : i + seq_len])
        # label aligned with the "end" of the current window
        y.append(labels[i + seq_len - 1])

    X = np.array(X, dtype=np.float32)
    y = np.array(y, dtype=np.int64)

    return X, y


############################################
# 2. TRANSFORMER MODEL
############################################

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]

        self.register_buffer('pe', pe)

    def forward(self, x):
        """
        x shape: [batch_size, seq_len, d_model]
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return x


class TransformerTimeSeriesModel(nn.Module):
    """
    A Transformer-based time series model which can handle sequences
    of feature vectors and output class logits.
    """
    def __init__(self, feature_dim=64, d_model=128, n_heads=4, num_layers=4,
                 dim_feedforward=256, dropout=0.1, num_classes=3,
                 use_cls_token=True):
        super().__init__()
        self.use_cls_token = use_cls_token

        # A learnable [CLS] token if we use it
        if self.use_cls_token:
            self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # Project input features into d_model
        self.input_projection = nn.Linear(feature_dim, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model=d_model, max_len=2000)

        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='relu',
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes)
        )

        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x):
        """
        x: [batch_size, seq_len, feature_dim]
        returns logits of shape [batch_size, num_classes]
        """
        batch_size, seq_len, feat_dim = x.size()
        # Project input to d_model dimension
        x = self.input_projection(x)  # => [B, L, d_model]

        # Optionally prepend a learnable [CLS] token
        if self.use_cls_token:
            cls_token = self.cls_token.repeat(batch_size, 1, 1)  # [B, 1, d_model]
            x = torch.cat([cls_token, x], dim=1)  # => [B, L+1, d_model]

        # Add positional encoding
        x = self.pos_encoder(x)

        # Pass through Transformer
        x = self.transformer_encoder(x)  # => [B, L or (L+1), d_model]

        # If we used a CLS token, take its final hidden state
        if self.use_cls_token:
            out = x[:, 0, :]
        else:
            # Otherwise, mean pool across time steps
            out = x.mean(dim=1)

        # Classification
        logits = self.classifier(out)
        return logits


############################################
# 3. TRAINING & EVALUATION
############################################

def train_one_epoch(model, dataloader, optimizer, criterion, device, clip_grad=1.0):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()

        # Gradient clipping to avoid exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += X_batch.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def eval_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(logits, dim=1)
            correct += (preds == y_batch).sum().item()
            total += X_batch.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def train_transformer(model,
                      train_loader,
                      val_loader,
                      criterion,
                      optimizer,
                      scheduler=None,
                      device='cuda',
                      epochs=10,
                      patience=3):
    """
    Trains the Transformer with optional scheduler & early stopping.
    Saves and reloads the best model based on validation accuracy.
    """
    best_val_acc = 0.0
    epochs_no_improve = 0

    for epoch in range(1, epochs + 1):
        # -- Train
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)

        # -- Validate
        val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

        # -- Scheduler step
        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(val_loss)
            else:
                scheduler.step()

        print(f"Epoch [{epoch}/{epochs}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc*100:.2f}% || "
              f"Val Loss: {val_loss:.4f}   | Val Acc: {val_acc*100:.2f}%")

        # -- Check if this is the best model so far
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), "best_transformer.pth")
            print("  [*] Model improved; saved to best_transformer.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print("  [!] Early stopping triggered.")
                break

    print(f"[Done] Best Validation Accuracy: {best_val_acc*100:.2f}%")
    return model

In [53]:
############################################
# In [2]: Load Data from Your Pipeline and Prepare Sliding Windows
############################################

# 1) Load the dataframe from your custom pipeline
df = train_pipeline.get_processed_df()  # e.g., includes columns and a 'label' col

# 2) Choose a sequence length. Adjust this based on your timeframe:
#    - For 1-minute bars, you might use seq_len = 256 or 512 to capture multiple hours or days.
#    - For 5-minute bars, seq_len = 128 can capture roughly one trading day (if ~ 78 bars in a day, go bigger if you want more history).
#    - For daily bars, you might use seq_len = 30 or 60 (to capture 1-2 months of data).
#    The larger the seq_len, the more context the model sees, but it also requires more memory.
seq_len = 128  # Example: if your timeframe is 5-minute bars

# 3) Create sliding windows
X, y = create_sliding_windows(df, seq_len=seq_len, label_col='Signal')
print("X shape:", X.shape)  # (N, seq_len, num_features)
print("y shape:", y.shape)  # (N,)

X shape: (10055, 128, 34)
y shape: (10055,)


In [54]:
############################################
# In [3]: Split into Training/Validation Sets
############################################

# Typical 80/20 split (time-based)
split_idx = int(len(X) * 0.8)
X_train, X_val = X[:split_idx], X[split_idx:]
y_train, y_val = y[:split_idx], y[split_idx:]

print("Train samples:", len(X_train), "Val samples:", len(X_val))

Train samples: 8044 Val samples: 2011


In [55]:
# ----------------------------------------------------
# MinMax Scaling PER FEATURE
# ----------------------------------------------------
# We'll do it per column so we can invert each column if needed.
# For time-series, typically you scale training portion, then apply the same
# scaling to validation. (Not re-fitting on validation data.)
num_features = X_train.shape[-1]

# We'll store one MinMaxScaler per feature column
minmax_scalers = []
for feat_idx in range(num_features):
    scaler = MinMaxScaler(feature_range=(0, 1))
    # Fit on training data for this feature
    scaler.fit(X_train[:, :, feat_idx].reshape(-1, 1))
    minmax_scalers.append(scaler)

    # Transform both train & val
    # shape => [n_samples, seq_len, 1] then we squeeze back
    X_train[:, :, feat_idx] = scaler.transform(X_train[:, :, feat_idx].reshape(-1, 1)).reshape(-1, seq_len)
    X_val[:, :, feat_idx] = scaler.transform(X_val[:, :, feat_idx].reshape(-1, 1)).reshape(-1, seq_len)

# Now X_train, X_val are scaled. We can invert them per column if needed
# using minmax_scalers[feat_idx].inverse_transform(...)

In [56]:
# Create PyTorch Datasets & DataLoaders
batch_size = 32
train_dataset = TimeSeriesDataset(X_train, y_train)
val_dataset   = TimeSeriesDataset(X_val, y_val)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader    = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

In [57]:
############################################
# In [4]: Instantiate and Train the Transformer Model
############################################

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# The feature_dim must match the number of features you have in X
num_features = X.shape[2]

model = TransformerTimeSeriesModel(
    feature_dim=num_features,   # match your # of input features
    d_model=128,                # embedding dimension
    n_heads=4,                  # number of attention heads
    num_layers=4,               # number of Transformer encoder layers
    dim_feedforward=256,
    dropout=0.2,
    num_classes=3,              # hold, buy, sell
    use_cls_token=True
).to(device)

# Attempt to load existing model weights if they exist
try:
    # Set weights_only=True to address the FutureWarning
    model.load_state_dict(torch.load("best_transformer.pth", weights_only=True))
    print("[Info] Existing 'best_transformer.pth' loaded. Continuing training...")
except FileNotFoundError:
    print("[Info] No existing model found. Training from scratch...")

Using device: cpu
[Info] No existing model found. Training from scratch...


In [58]:
# ----------------------------------------------------
# Define Loss, Optimizer, Scheduler
# ----------------------------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)

# Learning rate scheduler: reduce LR if val loss stops improving
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

In [ ]:
# Train
epochs = 5
model = train_transformer(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=epochs,
        patience=5  # Early stop if no improvement in 5 epochs
    )

In [ ]:
# ----------------------------------------------------
# Evaluate / Inference
# ----------------------------------------------------
# Load best saved model
try:
    # Set weights_only=True to address the FutureWarning
    model.load_state_dict(torch.load("best_transformer.pth", weights_only=True))
    print("[Info] 'best_transformer.pth' loaded successfully for evaluation.")
except FileNotFoundError:
    print("[Error] 'best_transformer.pth' not found. Ensure the model has been trained and saved correctly.")
except RuntimeError as e:
    print(f"[Error] An error occurred while loading the model: {e}")

# Set the model to evaluation mode
model.eval()

# Disable gradient calculation for evaluation
with torch.no_grad():
    val_loss, val_acc = eval_one_epoch(model, val_loader, criterion, device)

print(f"FINAL VAL LOSS: {val_loss:.4f} | FINAL VAL ACC: {val_acc*100:.2f}%")

In [ ]:
############################################
# In [6]: Example of Live/Real-Time Inference
############################################

# Simulate a "live" sample by just taking the first sample from the validation set
# In practice, you'd gather the latest seq_len bars from your feed, reshape them,
# and run it through the model.

sample_input = X_val[0]  # shape (seq_len, num_features)
sample_input = torch.from_numpy(sample_input).unsqueeze(0).float().to(device)

model.eval()
with torch.no_grad():
    logits = model(sample_input)  # shape [1, 3]
    probs = torch.softmax(logits, dim=1)
    pred_label = torch.argmax(probs, dim=1).item()

print("Live/Real-time predicted class =", pred_label)
print("Probabilities =", probs.squeeze().cpu().numpy())

In [ ]:
def backtest_model_with_targets(
    model,
    df,
    feature_cols,
    seq_len=128,
    close_col="close",
    high_col="high",
    low_col="low",
    target_col="Target",
    sl_col="StopLoss",
    initial_capital=10000.0,
    initial_quantity=15,
    brokerage_per_trade=20.0,
    device="cpu"
):
    """
    Backtest logic (one trade at a time):
      - On each bar i (starting from i=seq_len), gather the last seq_len bars [i-seq_len : i) into a [1, seq_len, num_features] tensor.
      - Feed this sequence to the model to get a predicted class: 0=hold, 1=buy, 2=sell.
      - If not in a trade and prediction is 'buy' or 'sell', we open a position at the current bar's Close.
      - Then on each subsequent bar, we check if the target or stop has been hit.
        * If triggered, we close the trade, update capital, pay brokerage, track P/L, etc.
        * We do not take new signals while in a trade.
      - If capital doubles from the last threshold, double the trade quantity.
      - Return various stats: final_capital, number of trades, win_rate, max_drawdown, and the capital_curve,
        plus a DataFrame (`trade_log`) containing a detailed row for each bar/time-step.
    """

    model.eval()

    capital = initial_capital
    current_quantity = initial_quantity
    # Next threshold for doubling position size:
    double_threshold = capital * 2.0

    peak_capital = capital  # track peak for drawdown
    max_drawdown = 0.0

    in_trade = False
    direction = 0        # +1 for buy, -1 for sell
    entry_price = 0.0
    target_price = 0.0
    stop_price = 0.0

    num_trades = 0
    wins = 0
    capital_curve = [capital]

    # For logging every bar:
    logs = []  # list of dicts, one per bar/time step

    i = seq_len
    n = len(df)

    while i < n:

        # Stop if we're out of capital
        if capital <= 0:
            print("[INFO] Capital exhausted; stopping backtest.")
            capital = 0
            break

        row_i = df.iloc[i]
        bar_time = row_i.name  # This could be the datetime or integer index

        # Prepare a dict to log this timestep's info.
        # We'll fill in more details below.
        log_entry = {
            "index": i,
            "timestamp": bar_time,
            "close": row_i[close_col],
            "in_trade": in_trade,
            "direction": direction,  # 0=flat, +1=long, -1=short
            "entry_price": entry_price if in_trade else None,
            "target_price": target_price if in_trade else None,
            "stop_price": stop_price if in_trade else None,
            "capital_before": capital,
            "action": "hold",  # default, will overwrite if buy/sell
            "reason_closed": None,
            "pnl_this_bar": 0.0,
            "capital_after": capital,  # will update if trade closes
        }

        if not in_trade:
            # ---------------------------
            #  Get model prediction
            # ---------------------------
            seq_df = df.iloc[i - seq_len : i][feature_cols]  # (seq_len, num_features)
            X_seq = torch.tensor(seq_df.values, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                logits = model(X_seq)               # => [1, 3]
                pred = torch.argmax(logits, dim=1)  # => [1]
                pred = pred.item()                  # => 0=hold,1=buy,2=sell

            if pred == 1:  # BUY
                in_trade = True
                direction = +1
                entry_price = row_i[close_col]
                target_price = entry_price + row_i[target_col]
                stop_price   = entry_price - row_i[sl_col]
                num_trades += 1

                print(f"[{i}] BUY Signal -> Entry: {entry_price}, Target: {target_price}, SL: {stop_price}")

                log_entry["action"] = "buy"
                log_entry["entry_price"] = entry_price
                log_entry["target_price"] = target_price
                log_entry["stop_price"] = stop_price

            elif pred == 2:  # SELL
                in_trade = True
                direction = -1
                entry_price = row_i[close_col]
                target_price = entry_price - row_i[target_col]
                stop_price   = entry_price + row_i[sl_col]
                num_trades += 1

                print(f"[{i}] SELL Signal -> Entry: {entry_price}, Target: {target_price}, SL: {stop_price}")

                log_entry["action"] = "sell"
                log_entry["entry_price"] = entry_price
                log_entry["target_price"] = target_price
                log_entry["stop_price"] = stop_price

            # If pred == 0 (hold), do nothing

            logs.append(log_entry)
            i += 1
            continue

        else:
            # ----------------------------------------------------
            #  Already in a trade; check if target or stop is hit
            # ----------------------------------------------------
            high_val = row_i[high_col]
            low_val  = row_i[low_col]

            trade_closed = False
            trade_pl_points = 0.0
            reason_closed = None

            if direction == +1:  # Long
                # If both target & stop occur in same bar:
                if (high_val >= target_price) and (low_val <= stop_price):
                    # Tie-break
                    open_val = row_i[close_col]  # or row_i["open"] if you have it
                    dist_target = abs(open_val - target_price)
                    dist_stop   = abs(open_val - stop_price)
                    if dist_target < dist_stop:
                        trade_pl_points = (target_price - entry_price)
                        reason_closed = "target_hit_same_bar"
                    else:
                        trade_pl_points = (stop_price - entry_price)
                        reason_closed = "stop_hit_same_bar"
                    trade_closed = True

                elif high_val >= target_price:
                    trade_pl_points = (target_price - entry_price)
                    reason_closed = "target_hit"
                    trade_closed = True

                elif low_val <= stop_price:
                    trade_pl_points = (stop_price - entry_price)
                    reason_closed = "stop_hit"
                    trade_closed = True

            else:  # direction == -1 (Short)
                # If both target & stop occur in same bar:
                if (low_val <= target_price) and (high_val >= stop_price):
                    open_val = row_i[close_col]
                    dist_target = abs(open_val - target_price)
                    dist_stop   = abs(open_val - stop_price)
                    if dist_target < dist_stop:
                        # target first
                        trade_pl_points = (entry_price - target_price)
                        reason_closed = "target_hit_same_bar"
                    else:
                        # stop first
                        trade_pl_points = (entry_price - stop_price)
                        reason_closed = "stop_hit_same_bar"
                    trade_closed = True

                elif low_val <= target_price:
                    trade_pl_points = (entry_price - target_price)
                    reason_closed = "target_hit"
                    trade_closed = True

                elif high_val >= stop_price:
                    trade_pl_points = (entry_price - stop_price)
                    reason_closed = "stop_hit"
                    trade_closed = True

            if trade_closed:
                # 1) Convert points to P/L in currency
                trade_pl_rupees = trade_pl_points * current_quantity
                # 2) Subtract brokerage
                trade_pl_rupees -= brokerage_per_trade
                # 3) Update capital
                old_capital = capital
                capital += trade_pl_rupees
                # 4) Win or loss?
                if trade_pl_rupees > 0:
                    wins += 1
                # 5) Track drawdown
                if capital > peak_capital:
                    peak_capital = capital
                drawdown = peak_capital - capital
                if drawdown > max_drawdown:
                    max_drawdown = drawdown

                print(
                    f"[{i}] Trade closed ({reason_closed}). "
                    f"P/L: {trade_pl_rupees:.2f}, New Capital: {capital:.2f}"
                )

                log_entry["action"] = "close_trade"
                log_entry["reason_closed"] = reason_closed
                log_entry["pnl_this_bar"] = trade_pl_rupees
                log_entry["capital_after"] = capital

                # 6) If capital <= 0, end
                if capital <= 0:
                    print("[INFO] Capital exhausted; stopping backtest.")
                    logs.append(log_entry)
                    break

                # 7) Record the new capital in the equity curve
                capital_curve.append(capital)

                # 8) Close the trade
                in_trade = False
                direction = 0
                entry_price = 0.0
                target_price = 0.0
                stop_price = 0.0

                # 9) Check if we have crossed the next doubling threshold
                if capital >= double_threshold:
                    current_quantity *= 2
                    double_threshold *= 2  # set next doubling level
            else:
                # If the trade is still open, no special action
                # but we'll log that no closure occurred
                log_entry["action"] = "in_trade"

            logs.append(log_entry)
            i += 1

    # --------------------
    # Compute final stats
    # --------------------
    final_capital = capital
    if num_trades > 0:
        win_rate = (wins / num_trades) * 100.0
    else:
        win_rate = 0.0

    if peak_capital > 0:
        max_drawdown_pct = (max_drawdown / peak_capital) * 100.0
    else:
        max_drawdown_pct = 0.0

    results = {
        "final_capital": final_capital,
        "num_trades": num_trades,
        "win_rate": win_rate,
        "max_drawdown": max_drawdown,
        "max_drawdown_%": max_drawdown_pct,
        "capital_curve": capital_curve,
        # Convert the logs into a DataFrame so you can analyze them after backtesting:
        "trade_log": pd.DataFrame(logs)
    }
    return results

In [ ]:
# Suppose your df has already been scaled with the same scalers used in training.
# Also ensure df contains columns: close, high, low, Target, StopLoss, and all feature_cols.

backtest_pipeline = FullFeaturePipeline(train_df, interval_minutes)
backtest_pipeline.run_pipeline()

results = backtest_model_with_targets(
    model=model,
    df=backtest_pipeline.get_processed_df(),               # scaled & pre-processed DataFrame
    feature_cols=backtest_pipeline.get_processed_df().columns.tolist(),  # same columns used in training
    seq_len=128,
    close_col="close",
    high_col="high",
    low_col="low",
    target_col="Target",
    sl_col="StopLoss",
    initial_capital=10000.0,
    initial_quantity=15,
    brokerage_per_trade=20.0,
    device=device
)

print("BACKTEST RESULTS:")
for k, v in results.items():
    if k != "capital_curve":
        print(f"{k}: {v}")

# Plot equity curve
import matplotlib.pyplot as plt

plt.plot(results["capital_curve"])
plt.title("Equity Curve")
plt.xlabel("Trades")
plt.ylabel("Capital")
plt.show()

In [ ]:
def get_sleep_time(interval_minutes, market_start_hour=9, market_start_minute=15, market_close_hour=15, market_close_minute=0):
    # Get current time in IST
    now = datetime.now(pytz.utc).astimezone(ist_timezone)

    # Define the market start and close times in IST for today
    market_start_time = now.replace(hour=market_start_hour, minute=market_start_minute, second=0, microsecond=0)
    market_close_time = now.replace(hour=market_close_hour, minute=market_close_minute, second=0, microsecond=0)

    if now < market_start_time:
        # If current time is before market starts, set next_run_time to market start time
        next_run_time = market_start_time
    elif now > market_close_time:
        # If current time is after market close, calculate the time until the next market open
        next_market_start_time = market_start_time + timedelta(days=1)
        next_run_time = next_market_start_time
    else:
        # Calculate the minutes since the market start time
        minutes_since_market_start = (now - market_start_time).total_seconds() // 60
        # Calculate the number of minutes to the next interval boundary
        minutes_to_next_interval = interval_minutes - (minutes_since_market_start % interval_minutes)
        # Calculate the next run time by adding these minutes to the current time
        next_run_time = (now + timedelta(minutes=minutes_to_next_interval)).replace(second=0, microsecond=0)

    # Calculate the sleep time in seconds
    sleep_time = (next_run_time - now).total_seconds()
    return sleep_time

In [ ]:
def fetch_option_chain():
    while True:
        try:
            data = {
                "symbol": index_symbol,
                "strikecount": 2,
                "timestamp": ""
            }
            response = fyers.optionchain(data=data)

            if response is not None:
                return response
        except Exception as e:
            print(f"Error fetching Option Chain: {e}")
            time.sleep(active_order_sleep)

index_oc= fetch_option_chain()

pd.DataFrame(index_oc['data']['optionsChain'])

In [ ]:
def assign_ce_pe_option_symbols():
    symbol_oc = fetch_option_chain()

    if symbol_oc != None:
        # Convert the response data into a DataFrame
        oc_df = pd.DataFrame(symbol_oc['data']['optionsChain'])

        # Find the first 'CE' symbol from the top
        first_ce_symbol = None
        for index, row in oc_df.iterrows():
            if row['option_type'] == 'CE':
                first_ce_symbol = row['symbol']
                first_ce_strike = row['strike_price']
                break

        # Find the first 'PE' symbol from the bottom
        first_pe_symbol = None
        for index, row in oc_df[::-1].iterrows():  # Iterate in reverse
            if row['option_type'] == 'PE':
                first_pe_symbol = row['symbol']
                first_pe_strike = row['strike_price']
                break

        return first_ce_symbol, first_pe_symbol, first_ce_strike, first_pe_strike

In [ ]:
def onmessage_ce(ce_message):
    global ce_ltp, index_ltp, unsubscribe_done
    try:
        if ce_message['symbol'] == ce_symbol:
            if "ltp" in ce_message:
                ce_ltp = ce_message["ltp"]
                ce_ltp = float(ce_ltp)

        elif ce_message['symbol'] == index_symbol:
            if "ltp" in ce_message:
                index_ltp = ce_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [ce_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {ce_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessageCE): {e}")


def onerror_ce(message):
    print("CE Error:", message)


def onclose_ce(message):
    print("CE Connection closed:", message)


def onopen_ce():

    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [ce_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def ce_buy_sell_ltp():
    global buy_sell_checked, ce_symbol, ce_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching CE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if ce_symbol is not None and ce_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                ce_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_ce,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_ce,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_ce,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_ce             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                ce_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def onmessage_pe(pe_message):
    global pe_ltp, index_ltp, unsubscribe_done
    try:
        if pe_message['symbol'] == pe_symbol:
            if "ltp" in pe_message:
                pe_ltp = pe_message["ltp"]
                pe_ltp = float(pe_ltp)

        elif pe_message['symbol'] == index_symbol:
            if "ltp" in pe_message:
                index_ltp = pe_message["ltp"]
                index_ltp = float(index_ltp)

        if sl_hit_condition and not unsubscribe_done:
            data_type = "SymbolUpdate"
            symbols_to_unsubscribe = [pe_symbol, index_symbol]
            fyers_socket.unsubscribe(symbols=symbols_to_unsubscribe, data_type=data_type)

            unsubscribe_done = True  # Set the flag to True after unsubscribing

            print(f"Unsubscribed {pe_symbol} & {index_symbol}")

    except Exception as e:
        print(f"Error (onMessagePE): {e}")

def onerror_pe(message):
    print("PE Error:", message)


def onclose_pe(message):
    print("PE Connection closed:", message)


def onopen_pe():
    # Specify the data type and symbols you want to subscribe to
    data_type = "SymbolUpdate"

    # Subscribe to the specified symbols and data type
    symbols = [pe_symbol, index_symbol]
    fyers_socket.subscribe(symbols=symbols, data_type=data_type)

    # Keep the socket running to receive real-time data
    fyers_socket.keep_running()

# Function to fetch and return the Call Option's Last Traded Price (LTP), strike, and symbol.
def pe_buy_sell_ltp():
    global buy_sell_checked, pe_symbol, pe_strike
    try:
        if not buy_sell_checked:
            buy_sell_checked = True

            print("Fetching PE Strike Price LTP")

            ce_symbol, pe_symbol, ce_strike, pe_strike = assign_ce_pe_option_symbols()

            if pe_symbol is not None and pe_strike is not None:
                # Create a FyersDataSocket instance with the provided parameters
                pe_socket_fyers = data_ws.FyersDataSocket(
                    access_token=ws_token,       # Access token in the format "appid:accesstoken"
                    log_path="",                     # Path to save logs. Leave empty to auto-create logs in the current directory.
                    litemode=True,                  # Lite mode disabled. Set to True if you want a lite response.
                    write_to_file=False,              # Save response in a log file instead of printing it.
                    reconnect=True,                  # Enable auto-reconnection to WebSocket on disconnection.
                    on_connect=onopen_pe,               # Callback function to subscribe to data upon connection.
                    on_close=onclose_pe,                # Callback function to handle WebSocket connection close events.
                    on_error=onerror_pe,                # Callback function to handle WebSocket errors.
                    on_message=onmessage_pe             # Callback function to handle incoming messages from the WebSocket.
                )

                # Establish a connection to the Fyers WebSocket
                pe_socket_fyers.connect()

    except Exception as e:
        print(f"Error fetching CE Buy/Sell LTP: {e}")

In [ ]:
def place_order(symbol):
    try:
        market_order_data = {
            "symbol": symbol,
            "qty": int(quantity),
            "type": 2,  # Market Order
            "side": 1,
            "productType": "INTRADAY",
            "limitPrice": 0,
            "stopPrice": 0,
            "validity": "DAY",
            "disclosedQty": 0,
            "offlineOrder":False
        }

        market_order_entry = fyers.place_order(data=market_order_data)

        if "id" in market_order_entry:
            market_order_id = market_order_entry["id"]
            market_order_message = market_order_entry["message"]
            print(f"{market_order_message}")

    except Exception as e:
        print(f"Error placing orders: {str(e)}")

In [ ]:
def trail_order(symbol, stoploss):
    while True:
        try:
            stoploss = int(stoploss)
            pending_order = fyers.orderbook()

            matching_orders = [order for order in pending_order["orderBook"] if order["status"] == 6]

            modified_orders = 0

            for order in matching_orders:
                if order['symbol'] == symbol:
                    pending_order_id = order['id']
                    pending_order_side = order['side']
                    pending_order_side = int(pending_order_side)

                    if pending_order_side != 1:
                        data = {
                            "id": pending_order_id,
                            "type": 4,
                            "limitPrice": stoploss - 1,
                            "stopPrice": stoploss
                        }

                        modify = fyers.modify_order(data=data)
                        trail_message = modify["message"]
                        print(f"{trail_message}")

                        if trail_message == "Successfully modified order":
                            modified_orders += 1

            # Check if all matching orders are successfully modified
            if modified_orders == len(matching_orders):
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print("Error modifying order:" + str(e))

In [ ]:
def exit_active_order(symbol):
    while True:
        try:
            data = {
                "id":f"{symbol}-INTRADAY"
            }

            exit_response = fyers.exit_positions(data=data)

            if ["message"] in exit_response:
                print(exit_response["message"])
                break

            time.sleep(active_order_sleep)

        except Exception as e:
            print(f"Error exiting Order: {e}")

In [ ]:
def reset_flags():
    global active_order, buy_sell_checked

    active_order = False
    buy_sell_checked = False

In [ ]:
# Function to save profits and losses
def save_overall(overall_win, overall_loss, capital):
    trade_type = {
        "overall_win": overall_win,
        "overall_loss": overall_loss,
        "capital": capital
    }

    with open("trade_data.json", "w") as file:
        json.dump(trade_type, file)


# Function to load wins and losses
def load_overall():
    try:
        with open('trade_data.json') as file:
            return json.load(file)
    except FileNotFoundError:
        return None

In [ ]:
def handle_active_ce_order():
    def ce_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, ce_ltp, index_ltp, ce_strike, ce_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        ce_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if ce_ltp != 0 and index_ltp != 0:
                    ce_ltp_array.append(ce_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)


                    if index_ltp <= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(ce_symbol)

                        points = int(ce_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("CE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp >= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                            trailing_sl_inside = int(ce_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp - stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(ce_ltp + target)
                            target_index_inside = int(index_ltp + target)

                    else:
                        if (index_ltp - prev_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(ce_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp - trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(ce_ltp_array, label=f"LTP: {ce_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("CE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=ce_order_loop()).start()

In [ ]:
def handle_active_pe_order():
    def pe_order_loop():
        global prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, total_profit, total_loss, overall_win, overall_loss, pe_ltp, index_ltp, pe_strike, pe_symbol, sl_hit_condition, total_points, fixed_ltp, fixed_index_ltp, target, trailing_sl

        profit_money = 0
        overall_win = 0
        overall_loss = 0
        capital = 0
        profit_percentage = 0
        loss_percentage = 0

        pe_ltp_array = []
        index_ltp_array = []

        target_hit_once = False

        while True:
            try:
                if pe_ltp != 0 and index_ltp != 0:
                    pe_ltp_array.append(pe_ltp)
                    index_ltp_array.append(index_ltp)

                    trade_data = load_overall()

                    if trade_data:
                        overall_win = trade_data["overall_win"]
                        overall_loss = trade_data["overall_loss"]
                        capital = trade_data["capital"]

                    total_trades = overall_win + overall_loss

                    if overall_win > 0:
                        profit_percentage = (overall_win/total_trades) * 100
                        profit_percentage = round(profit_percentage, 2)

                    if overall_loss > 0:
                        loss_percentage = (overall_loss/total_trades) * 100
                        loss_percentage = round(loss_percentage, 2)

                    if index_ltp >= trailing_index_inside:
                        pygame.mixer.music.play()

                        #exit_active_order(pe_symbol)

                        points = int(pe_ltp) - int(fixed_ltp)

                        total_points = total_points + points

                        if points > 0:
                            total_profit += 1
                            overall_win += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)


                        elif points < 0:
                            total_loss += 1
                            overall_loss += 1
                            total_trades = overall_win + overall_loss

                            if overall_win > 0:
                                profit_percentage = (overall_win/total_trades) * 100
                                profit_percentage = round(profit_percentage, 2)

                            if overall_loss > 0:
                                loss_percentage = (overall_loss/total_trades) * 100
                                loss_percentage = round(loss_percentage, 2)

                        profit_money = points*quantity
                        capital = (capital + profit_money) - brokerage
                        save_overall(overall_win, overall_loss, capital)

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        clear_output(wait=True)

                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                        plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                        plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                        plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("PE Order LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display profit/loss information as text annotations
                        plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                        plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        # Apply dark background style to Matplotlib
                        plt.style.use('dark_background')

                        # Plotting
                        plt.figure(figsize=(14, 7))

                        # Plot LTP data with labels
                        plt.plot(index_ltp_array, label=f"LTP: {index_ltp_array}", color='white')
                        plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                        plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                        plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                        # Set labels and title
                        plt.xlabel("Time", color='white')
                        plt.ylabel("LTP", color='white')
                        plt.title("Index LTP Chart", color='white')

                        # Customize legend
                        plt.legend(facecolor='black')

                        # Remove grid lines
                        plt.grid(False)

                        # Display the plot
                        display(plt.gcf())
                        plt.close()

                        sl_hit_condition = True

                        reset_flags()

                        break

                    elif index_ltp <= target_index_inside:
                        pygame.mixer.music.play()

                        if not target_hit_once:
                            target_hit_once = True
                            target /= 2
                            stop_loss =  trailing_sl / 10

                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                            trailing_sl_inside = int(pe_ltp - stop_loss)
                            trailing_index_inside = int(index_ltp + stop_loss)

                            trailing_sl /= 2
                            prev_ltp = trailing_index_inside

                        else:
                            target_inside = int(pe_ltp + target)
                            target_index_inside = int(index_ltp - target)

                    else:
                        if (prev_ltp - index_ltp) >= trailing_sl and target_hit_once:
                            pygame.mixer.music.play()

                            prev_ltp = index_ltp

                            trailing_sl_inside = int(pe_ltp - trailing_sl)
                            trailing_index_inside = int(index_ltp + trailing_sl)

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    clear_output(wait=True)

                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(pe_ltp_array, label=f"LTP: {pe_ltp}", color='white')
                    plt.axhline(y=fixed_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_ltp}')
                    plt.axhline(y=target_inside, color='green', linestyle='-', label=f'Target: {target_inside}')
                    plt.axhline(y=trailing_sl_inside, color='red', linestyle='-', label=f'SL: {trailing_sl_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("PE Order LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display profit/loss information as text annotations
                    plt.text(0.02, 0.95, f"Total Profit: {total_profit}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.90, f"Total Loss: {total_loss}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.85, f"Points Captured: {total_points}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.80, f"Profit/Loss: {profit_money}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.75, f"Overall Profit: {overall_win} / {profit_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.70, f"Overall Loss: {overall_loss} / {loss_percentage}%", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')
                    plt.text(0.02, 0.65, f"Capital: {capital}", transform=plt.gca().transAxes, fontsize=12, color='white', verticalalignment='top')

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                    # Apply dark background style to Matplotlib
                    plt.style.use('dark_background')

                    # Plotting
                    plt.figure(figsize=(14, 7))

                    # Plot LTP data with labels
                    plt.plot(index_ltp_array, label=f"LTP: {index_ltp}", color='white')
                    plt.axhline(y=fixed_index_ltp, color='blue', linestyle='--', label=f'Entry LTP: {fixed_index_ltp}')
                    plt.axhline(y=target_index_inside, color='green', linestyle='-', label=f'Target: {target_index_inside}')
                    plt.axhline(y=trailing_index_inside, color='red', linestyle='-', label=f'SL: {trailing_index_inside}')

                    # Set labels and title
                    plt.xlabel("Time", color='white')
                    plt.ylabel("LTP", color='white')
                    plt.title("Index LTP Chart", color='white')

                    # Customize legend
                    plt.legend(facecolor='black')

                    # Remove grid lines
                    plt.grid(False)

                    # Display the plot
                    display(plt.gcf())
                    plt.close()

                time.sleep(active_order_sleep)

            except Exception as e:
                print(f"Error: {e}")

    threading.Thread(target=pe_order_loop()).start()

In [ ]:
def ce_entry():
    threading.Thread(target=ce_buy_sell_ltp).start()

    def ce_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if ce_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = ce_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp + target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp - trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(ce_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_ce_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=ce_entry_thread()).start()

In [ ]:
def pe_entry():
    threading.Thread(target=pe_buy_sell_ltp).start()

    def pe_entry_thread():
        global fixed_ltp, fixed_index_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, active_order, prev_ltp

        pygame.mixer.music.play()

        while True:
            if pe_ltp != 0 and index_ltp !=0:
                prev_ltp = index_ltp
                temp_index_ltp = prev_ltp

                temp_ltp = pe_ltp

                target_inside = temp_ltp + target
                target_inside = int(target_inside)

                trailing_sl_inside = temp_ltp - trailing_sl
                trailing_sl_inside = int(trailing_sl_inside)

                target_index_inside = temp_index_ltp - target
                target_index_inside = int(target_index_inside)

                trailing_index_inside = temp_index_ltp + trailing_sl
                trailing_index_inside = int(trailing_index_inside)

                if target_inside <= 0:
                    target_inside = 0
                if trailing_sl_inside <= 0:
                    trailing_sl_inside = 0

                #place_order(pe_symbol)

                fixed_ltp = temp_ltp
                fixed_index_ltp = temp_index_ltp

                active_order = True

                handle_active_pe_order()

                break

            else:
                time.sleep(active_order_sleep)


    threading.Thread(target=pe_entry_thread()).start()

In [ ]:
def market_entry_exit_logic(action, actual_closing_price, final_df):
    global sl_hit_condition, unsubscribe_done, ce_ltp, pe_ltp, index_ltp, fixed_ltp, fixed_index_ltp, prev_ltp, target_inside, target_index_inside, trailing_sl_inside, trailing_index_inside, ce_strike, pe_strike, ce_symbol, pe_symbol

    ce_ltp = 0
    pe_ltp  =0
    index_ltp = 0
    fixed_ltp = 0
    fixed_index_ltp = 0
    prev_ltp = 0
    target_inside = 0
    target_index_inside = 0
    trailing_sl_inside = 0
    trailing_index_inside = 0
    ce_strike = None
    pe_strike = None
    ce_symbol = None
    pe_symbol = None

    final_current_time = final_df.index[-1].time()
    print(final_current_time)

    # Ensure trading occurs only during market hours
    if final_current_time >= dt_time(9, (15 + interval_minutes)) and final_current_time <= dt_time(18, 0):
        if action == 1 and not active_order:  # Buy action
            print("Buy action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making CE Position at {actual_closing_price}")
            ce_entry()  # Execute CE entry logic
        elif action == 2 and not active_order:  # Sell action
            print("Sell action detected by PPO model.")
            pygame.mixer.music.play()
            sl_hit_condition = False
            unsubscribe_done = False
            print(f"Making PE Position at {actual_closing_price}")
            pe_entry()  # Execute PE entry logic
        elif action == 0:
            print("Hold action detected by PPO model. No trade executed.")

In [ ]:
def fetch_and_prepare_final_data():
    final_df = fetch_candle_data(10)

    final_data_processor = DataProcessor(final_df, live_processing=True)

    final_processed_df = final_data_processor.process().df

    return final_processed_df

In [ ]:
def find_local_extrema(df):
    order=5
    atr_multiplier=1.5
    min_distance=5

    # Find local maxima and minima
    local_max = argrelextrema(df['high'].values, np.greater_equal, order=order)[0]
    local_min = argrelextrema(df['low'].values, np.less_equal, order=order)[0]

    # Calculate the threshold based on ATR
    threshold = df['atr'] * atr_multiplier

    # Filter by significance
    significant_max = []
    significant_min = []

    for idx in local_max:
        if idx > order and idx < len(df) - order:
            high = df['high'].iloc[idx]
            if significant_min:
                low = df['low'].iloc[significant_min[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_max.append(idx)
            else:
                significant_max.append(idx)

    for idx in local_min:
        if idx > order and idx < len(df) - order:
            low = df['low'].iloc[idx]
            if significant_max:
                high = df['high'].iloc[significant_max[-1]]
                if (high - low) > threshold.iloc[idx]:
                    significant_min.append(idx)
            else:
                significant_min.append(idx)

    # Ensure minimum distance
    def filter_by_distance(points, min_distance):
        filtered_points = []
        for i in range(len(points)):
            if not filtered_points or (points[i] - filtered_points[-1]) > min_distance:
                filtered_points.append(points[i])
        return filtered_points

    significant_max = filter_by_distance(significant_max, min_distance)
    significant_min = filter_by_distance(significant_min, min_distance)

    return significant_max, significant_min

In [ ]:
def get_trendline(df, point1, point2, kind='high'):
    x = [point1, point2]
    if kind == 'high':
        y = df['high'].values[x]
    else:
        y = df['low'].values[x]
    coeffs = np.polyfit(x, y, 1)
    trendline = np.polyval(coeffs, range(len(df)))
    return trendline

In [ ]:
# Online training configuration
online_learning_steps = 1000  # Steps for each retraining
retrain_interval_minutes = 60  # Retrain the model every 60 minutes

start_time = datetime.now()

while True:
    clear_output(wait=True)

    num_candles = 100

    final_df = fetch_and_prepare_final_data()

    final_df = final_df.iloc[:-1]

    target = final_df['Target'].iloc[-1]
    trailing_sl = final_df['Stop Loss'].iloc[-1]

    # Initialize the TradingEnvironment with the latest data
    live_env = create_env(final_df, config)
    obs = live_env.reset()

    # Use PPO model to predict action for the latest data point
    action, _ = ppo_model.predict(obs, deterministic=True)

    # Extract the actual closing price
    actual_closing_price = final_df['close'].iloc[-1]

    # Retrain PPO model at specified intervals
    if (datetime.now() - start_time).seconds >= retrain_interval_minutes * 60:
        print("Retraining PPO model with online data...")
        train_env = TradingEnvironment(final_df, config)  # Create a new training environment
        ppo_model.set_env(train_env)  # Update the PPO model's environment
        ppo_model.learn(total_timesteps=online_learning_steps)  # Retrain the model
        ppo_model.save(model_save_path)  # Save the updated model
        start_time = datetime.now()

    # Identify most recent high and low points
    recent_highs, recent_lows = find_local_extrema(final_df)

    most_recent_high = recent_highs[-1] if len(recent_highs) > 1 else None
    most_recent_low = recent_lows[-1] if len(recent_lows) > 1 else None

    high_trendline = [np.nan] * len(final_df)
    low_trendline = [np.nan] * len(final_df)

    if most_recent_high is not None:
        previous_high = recent_highs[-2] if len(recent_highs) > 2 else most_recent_high
        high_trendline = get_trendline(final_df, previous_high, most_recent_high, kind='high')

    if most_recent_low is not None:
        previous_low = recent_lows[-2] if len(recent_lows) > 2 else most_recent_low
        low_trendline = get_trendline(final_df, previous_low, most_recent_low, kind='low')

    # Prepare candlestick data for mplfinance
    actual_candles = final_df[-num_candles:].copy()

    # Create a DataFrame for mplfinance
    mpf_df = actual_candles[['open', 'high', 'low', 'close']]

    # Create addplot elements for predicted prices and actual close prices
    ap = [
        mpf.make_addplot(final_df['close'][-num_candles:], color='none', panel=0, secondary_y=False, label=f"Actual Price: {final_df['close'].iloc[-1]}"),
        #mpf.make_addplot(y_pred_ensemble_final_plot, color=(0.95, 0.38, 0.25, 1), panel=0, secondary_y=False, label=f'Predicted Prices ({y_pred_ensemble_final_plot[-1]:.2f})')
    ]

    # Add trendlines to the plot
    if most_recent_high is not None:
        ap.append(mpf.make_addplot(high_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    if most_recent_low is not None:
        ap.append(mpf.make_addplot(low_trendline[-num_candles:], color='white', linestyle='-', panel=0, secondary_y=False))

    fig, axlist = mpf.plot(mpf_df, type='candle', style='binancedark', volume=False, addplot=ap,
                        title=f'Chart', ylabel='Price',
                        figsize=(14, 7), returnfig=True)

    for ax in axlist:
        ax.grid(False)

    axlist[0].legend()
    plt.show()

    # Execute market entry/exit logic
    market_entry_exit_logic(action, actual_closing_price, final_df)

    sleep_time = get_sleep_time(interval_minutes)
    print(f"Sleeping for {sleep_time}")
    time.sleep(sleep_time)